# Plant Disease Detection Data Preprocessing (32x32 Grayscale)

Bu notebook, "Plant Disease Detection" veri setini (Kaggle dataset)
klasik makine öğrenmesi yöntemleri için hazırlar.

Adımlar PlantVillage preprocessing ile aynıdır:
1. Klasör yapısındaki sınıfları okumak.
2. Görselleri 32x32 gri tonlamalı hale getirip [0, 1] aralığına normalize etmek.
3. Her sınıf için train / validation / test split yapmak.
4. İşlenmiş veri setini `preprocessed_pdd/*.npy` olarak kaydetmek.



# Cell 1 – Importlar ve ayarlar

In [1]:
import os
import random
from PIL import Image
import numpy as np

# -------------------
# Configuration
# -------------------

# Root for Plant Disease Detection dataset
# Directory structure:
#   ../Dataset/plantdiseasedetection/
#       ClassA/
#       ClassB/
#       ...
DATA_ROOT = "../Dataset/plantdiseasedetection"

OUTPUT_DIR = "preprocessed_pdd"

IMG_SIZE = 32
RANDOM_SEED = 42

TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
MAX_IMAGES_PER_CLASS = None

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# Cell 2 – Yardımcı fonksiyonlar (sınıf isimleri + görüntü vektörü)

In [2]:
def get_class_names(root_dir):
    class_names = []
    for item in os.listdir(root_dir):
        full_path = os.path.join(root_dir, item)
        if os.path.isdir(full_path):
            class_names.append(item)
    class_names = sorted(class_names)
    return class_names


def load_image_as_vector(path, img_size=IMG_SIZE):
    with Image.open(path) as img:
        img = img.convert("L")
        img = img.resize((img_size, img_size))
        pixels = list(img.getdata())
        vector = [p / 255.0 for p in pixels]
        return vector


# Cell 3 – Dataset’i oluşturma (train/val/test split)

In [3]:
def build_splits_for_dataset(root_dir, train_ratio, val_ratio, max_per_class=None):
    class_names = get_class_names(root_dir)
    print("Found classes:")
    for idx, name in enumerate(class_names):
        print(f"{idx:2d} -> {name}")
    print()

    X_train, y_train = [], []
    X_val, y_val = [], []
    X_test, y_test = [], []

    for class_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(root_dir, class_name)
        image_files = [
            f for f in os.listdir(class_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

        if max_per_class is not None:
            random.shuffle(image_files)
            image_files = image_files[:max_per_class]
        else:
            random.shuffle(image_files)

        n_total = len(image_files)
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)
        n_test = n_total - n_train - n_val

        train_files = image_files[:n_train]
        val_files = image_files[n_train:n_train + n_val]
        test_files = image_files[n_train + n_val:]

        print(f"Class '{class_name}' (idx {class_idx}): "
              f"total={n_total}, train={len(train_files)}, "
              f"val={len(val_files)}, test={len(test_files)}")

        for fname in train_files:
            path = os.path.join(class_dir, fname)
            try:
                x_vec = load_image_as_vector(path)
            except Exception as e:
                print(f"Warning: cannot load {path}: {e}")
                continue
            X_train.append(x_vec)
            y_train.append(class_idx)

        for fname in val_files:
            path = os.path.join(class_dir, fname)
            try:
                x_vec = load_image_as_vector(path)
            except Exception as e:
                print(f"Warning: cannot load {path}: {e}")
                continue
            X_val.append(x_vec)
            y_val.append(class_idx)

        for fname in test_files:
            path = os.path.join(class_dir, fname)
            try:
                x_vec = load_image_as_vector(path)
            except Exception as e:
                print(f"Warning: cannot load {path}: {e}")
                continue
            X_test.append(x_vec)
            y_test.append(class_idx)

    print("\nTotal samples:")
    print("Train:", len(X_train))
    print("Val  :", len(X_val))
    print("Test :", len(X_test))

    return (
        np.array(X_train, dtype=np.float32),
        np.array(y_train, dtype=np.int64),
        np.array(X_val, dtype=np.float32),
        np.array(y_val, dtype=np.int64),
        np.array(X_test, dtype=np.float32),
        np.array(y_test, dtype=np.int64),
        class_names,
    )


# Cell 4 – Preprocess’i çalıştır

In [4]:
(
    X_train_pdd,
    y_train_pdd,
    X_val_pdd,
    y_val_pdd,
    X_test_pdd,
    y_test_pdd,
    class_names_pdd,
) = build_splits_for_dataset(
    DATA_ROOT,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    max_per_class=MAX_IMAGES_PER_CLASS,
)

print("\nShapes:")
print("X_train:", X_train_pdd.shape, "y_train:", y_train_pdd.shape)
print("X_val  :", X_val_pdd.shape, "y_val  :", y_val_pdd.shape)
print("X_test :", X_test_pdd.shape, "y_test :", y_test_pdd.shape)


Found classes:
 0 -> Apple___Apple_scab
 1 -> Apple___Black_rot
 2 -> Apple___Cedar_apple_rust
 3 -> Apple___healthy
 4 -> Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
 5 -> Corn_(maize)___Common_rust_
 6 -> Corn_(maize)___Northern_Leaf_Blight
 7 -> Corn_(maize)___healthy
 8 -> Pepper__bell___Bacterial_spot
 9 -> Pepper__bell___healthy
10 -> Potato___Early_blight
11 -> Potato___Late_blight
12 -> Potato___healthy
13 -> Tomato_Bacterial_spot
14 -> Tomato_Early_blight
15 -> Tomato_Late_blight
16 -> Tomato_Leaf_Mold
17 -> Tomato_Septoria_leaf_spot
18 -> Tomato_Spider_mites_Two_spotted_spider_mite
19 -> Tomato__Target_Spot
20 -> Tomato__Tomato_YellowLeaf__Curl_Virus
21 -> Tomato__Tomato_mosaic_virus
22 -> Tomato_healthy

Class 'Apple___Apple_scab' (idx 0): total=2016, train=1411, val=302, test=303
Class 'Apple___Black_rot' (idx 1): total=1987, train=1390, val=298, test=299
Class 'Apple___Cedar_apple_rust' (idx 2): total=1760, train=1232, val=264, test=264
Class 'Apple___healthy' (idx 

# Cell 5 – .npy ve sınıf isimlerini kaydet

In [5]:
# Save arrays
np.save(os.path.join(OUTPUT_DIR, "train_X.npy"), X_train_pdd)
np.save(os.path.join(OUTPUT_DIR, "train_y.npy"), y_train_pdd)
np.save(os.path.join(OUTPUT_DIR, "val_X.npy"), X_val_pdd)
np.save(os.path.join(OUTPUT_DIR, "val_y.npy"), y_val_pdd)
np.save(os.path.join(OUTPUT_DIR, "test_X.npy"), X_test_pdd)
np.save(os.path.join(OUTPUT_DIR, "test_y.npy"), y_test_pdd)

# Save class names
class_names_path = os.path.join(OUTPUT_DIR, "class_names.txt")
with open(class_names_path, "w", encoding="utf-8") as f:
    for name in class_names_pdd:
        f.write(name + "\n")

print(f"\nSaved preprocessed Plant Disease Detection data to '{OUTPUT_DIR}'")
print(f"Class names written to {class_names_path}")


Saved preprocessed Plant Disease Detection data to 'preprocessed_pdd'
Class names written to preprocessed_pdd\class_names.txt
